# P1 / 01: daily reversal by abnormal turnover (faithful version)

Protocol: [`001_claude_p1_turnover_reversal_preregistration.md`](../../discussions/Short_Term_Momentum/001_claude_p1_turnover_reversal_preregistration.md), arm D. Paper: Medhat & Schmeling (2022), *Short-term Momentum*.

- **Turnover proxy:** abnormal volume AV = 20-session mean volume / 250-session mean volume (≥ 200), known at the close.
- **Sort:** AV terciles (T1 low … T3 high), then signal quintiles within each tercile. Reversal portfolio = long the bottom quintile, short the top, equal weight, +1 / −1.
- **Horizons:** past return over k ∈ {21 (primary), 5, 1} sessions, overlapping holding h = k.
- **Timing:** signal at the close of d, trade at the close of d+1, P&L from then on.
- **Costs:** S05b measured half-spread at the 15:45 slot × |Δw|.
- **Gates** (α = 0.05/3 per horizon): D1 T1 − T3 gross > 0 (bootstrap and shuffled-label placebo); D2 T3 gross < 0; D3 T1 net > 0 at 1× (bootstrap) and 2× (mean).

In [ ]:
N_JOBS = -1
PER_YEAR, WINDOW_S = 4, 3.0          # S05b quote samples (reused from cache)

import json, os, sys, time
from dataclasses import replace
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "equity_trading").exists())
sys.path.insert(0, str(ROOT / "src"))
from equity_trading import __version__, p1, s04, s05, s05b
from equity_trading.alpaca import ALPACA_VERSION, download_years, sample_quote_spreads
from equity_trading.data import vendor_dir, work_dir

SYNTHETIC = os.environ.get("P1_SYNTHETIC") == "1"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e6e6e3", "grid.linewidth": 0.6, "axes.titleweight": "bold", "font.size": 9})
GCOL = {0: "#2a78d6", 1: "#8a8a84", 2: "#eb6834"}          # T1 low AV, T2, T3 high AV
GNAME = {0: "T1 low AV", 1: "T2", 2: "T3 high AV"}

CFG = p1.P1Config()
YEARS_D = list(range(2019, 2026))                     # 2019: volume warm-up only
YEARS_Q = list(range(2020, 2026))                     # S05b quote-sample years; 2026 not used
if SYNTHETIC:
    CFG = replace(CFG, n_label_placebo=20, n_boot=2000)
tag = ("synthetic_" if SYNTHETIC else "") + f"alpaca{ALPACA_VERSION}_p1{p1.P1_VERSION}"
OUT = ROOT / "notebooks" / "Short_Term_Momentum" / ("01_synthetic" if SYNTHETIC else "01")
OUT.mkdir(parents=True, exist_ok=True)
CACHE = work_dir() / "results" / "p1" / tag
CACHE.mkdir(parents=True, exist_ok=True)
QUOTE_CACHE = vendor_dir("Alpaca") / "quote_samples" / f"15Min_w{WINDOW_S:g}s"
if not SYNTHETIC:
    constituents = pd.read_csv(ROOT / "src" / "equity_trading" / "reference" / "sp500_constituents_20260913.csv")
    STOCKS = sorted(constituents["Symbol"])
print(f"synthetic={SYNTHETIC} | horizons={CFG.horizons} | alpha={CFG.alpha:.4f} | label placebos={CFG.n_label_placebo}")

## 1. Data: daily bars, measured costs

In [ ]:
t0 = time.time()
if SYNTHETIC:
    C, O, V = p1.synthetic_daily(n_stocks=150, years=[2019, 2020, 2021], effect=0.15)
    COST = pd.DataFrame(3e-4, index=C.index, columns=C.columns)
    CFG = replace(CFG, trade_end="2021-12-31")
else:
    download_years(STOCKS, "1Day", YEARS_D, workers=8)
    C, O, V = p1.load_daily(STOCKS, YEARS_D)
    C, O, V = (x.dropna(axis=1, how="all") for x in (C, O, V))
    SESSIONS = s05b.sample_sessions(YEARS_Q, PER_YEAR, seed=42)
    missing = [d for d in SESSIONS if not (QUOTE_CACHE / f"{d}.parquet").exists()]
    if missing:                                         # normally cached by S05b
        QUOTE_CACHE.mkdir(parents=True, exist_ok=True)
        for d in missing:
            sample_quote_spreads(list(C.columns), d, s05b.slot_starts(YEARS_Q, d), window_s=WINDOW_S, cache_dir=QUOTE_CACHE)
    samples = pd.concat([pd.read_parquet(QUOTE_CACHE / f"{d}.parquet") for d in SESSIONS], ignore_index=True)
    COST = p1.daily_cost(C.index, list(C.columns), s05b.cost_model(samples))
RET = C.pct_change()
WIN = C.index[(C.index >= CFG.trade_start) & (C.index <= CFG.trade_end)]
print(f"{C.shape[1]} stocks, {len(C)} sessions ({C.index[0].date()} to {C.index[-1].date()}), "
      f"trading window {WIN[0].date()} to {WIN[-1].date()} ({len(WIN)} sessions); "
      f"close-slot cost median {1e4 * np.nanmedian(COST.loc[WIN].to_numpy()):.2f} bps one-way; ready in {time.time() - t0:.0f}s")

## 2. Abnormal volume and terciles

AV measures turnover relative to each stock's own norm, so terciles should not be a size sort. Persistence shows how long a stock stays in its tercile.

In [ ]:
AV = p1.abnormal_volume(V, CFG)
PRICE_OK = C >= CFG.min_price
lab_any = p1.group_labels(AV, CFG.n_groups, eligible=PRICE_OK)
dv = np.log((C * V).rolling(20).mean())                 # dollar volume, descriptive only
rows = []
for g in range(CFG.n_groups):
    m = (lab_any == g).loc[WIN]
    later = lab_any.shift(-21).loc[WIN]
    rows.append({"tercile": GNAME[g], "mean_AV": float(AV.loc[WIN][m].stack().mean()),
                 "median_log_dollar_volume": float(dv.loc[WIN][m].stack().median()),
                 "same_tercile_21d_later": float((later[m] == g).stack().mean()),
                 "names_per_day": float(m.sum(axis=1).mean())})
TERC = pd.DataFrame(rows).set_index("tercile")
TERC.to_csv(OUT / "terciles.csv", float_format="%.4g")
display(TERC.round(3))
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(np.log(AV.loc[WIN].stack().clip(0.05, 20)), bins=80, color="#2a78d6")
ax.set_title("log abnormal volume (all stock-days)"); plt.tight_layout(); plt.show()

## 3. Reversal portfolios by tercile

Daily returns per $1 long and $1 short. `net` is at 1× measured cost; 2× is `gross − 2 × cost`.

In [ ]:
RES, SIGS, LABS, TABLE = {}, {}, {}, []
for k in CFG.horizons:
    sig = np.log(C / C.shift(k))
    lab = p1.group_labels(AV, CFG.n_groups, eligible=PRICE_OK & sig.notna())
    SIGS[k], LABS[k] = sig, lab
    RES[k] = p1.arm_d_horizon(sig, lab, RET, COST, k, CFG, WIN)
    for g, df in RES[k].items():
        sd = df.std()
        TABLE.append({"k": k, "tercile": GNAME[g], "gross_bps_day": 1e4 * df.gross.mean(), "cost_bps_day": 1e4 * df.cost.mean(),
                      "net_bps_day": 1e4 * df.net.mean(), "net2x_bps_day": 1e4 * (df.gross - 2 * df.cost).mean(),
                      "sharpe_gross": np.sqrt(252) * df.gross.mean() / sd.gross if sd.gross > 0 else np.nan,
                      "sharpe_net": np.sqrt(252) * df.net.mean() / sd.net if sd.net > 0 else np.nan})
    pd.concat({GNAME[g]: df for g, df in RES[k].items()}, axis=1).to_csv(OUT / f"daily_returns_k{k}.csv", float_format="%.6g")
TABLE = pd.DataFrame(TABLE).set_index(["k", "tercile"])
display(TABLE.round(3))
fig, ax = plt.subplots(1, len(CFG.horizons), figsize=(4.2 * len(CFG.horizons), 3.4), sharey=False)
for a, k in zip(np.atleast_1d(ax), CFG.horizons):
    for g, df in RES[k].items():
        a.plot(df.index, df.gross.cumsum(), color=GCOL[g], lw=0.9, label=GNAME[g])
        a.plot(df.index, df.net.cumsum(), color=GCOL[g], lw=0.7, ls="--")
    a.axhline(0, color="#444", lw=0.6); a.set_title(f"k = {k}: cumulative return (solid gross, dashed net)")
np.atleast_1d(ax)[0].legend(frameon=False); plt.tight_layout(); plt.show()

## 4. Bootstrap, shuffled-label placebo, gates

In [ ]:
from joblib import Parallel, delayed
G, BOOT, PLACEBO = {}, {}, {}
hi = CFG.n_groups - 1
for k in CFG.horizons:
    r0, r2 = RES[k][0], RES[k][hi]
    diff = (r0.gross - r2.gross).to_numpy()
    b = lambda x, j: p1.block_bootstrap(x, j, CFG.n_boot, CFG.seed, CFG.alpha)  # noqa: E731
    BOOT[k] = {"diff_gross": {j: b(diff, j) for j in CFG.blocks},
               "high_gross": {j: b(r2.gross.to_numpy(), j) for j in CFG.blocks},
               "low_net": {j: b(r0.net.to_numpy(), j) for j in CFG.blocks}}
    pl_path = CACHE / f"label_placebo_k{k}_{CFG.n_label_placebo}.npy"
    if pl_path.exists():
        dist = np.load(pl_path)
    else:
        t0 = time.time()
        chunks = [(i, min(25, CFG.n_label_placebo - i)) for i in range(0, CFG.n_label_placebo, 25)]
        parts = Parallel(n_jobs=N_JOBS)(delayed(p1.label_placebo)(SIGS[k], LABS[k], RET, k, CFG, WIN, n, CFG.seed * 1000 + k * 100 + i)
                                         for i, n in chunks)
        dist = np.concatenate(parts); np.save(pl_path, dist)
        print(f"k={k}: {len(dist)} label placebos in {(time.time() - t0) / 60:.1f} min")
    obs = float(diff.mean())
    PLACEBO[k] = s04.placebo_rank(obs, dist)
    G[k] = p1.gates_d(BOOT[k]["diff_gross"], PLACEBO[k]["p_upper"], BOOT[k]["high_gross"], BOOT[k]["low_net"],
                      float((r0.gross - 2 * r0.cost).mean()), CFG.alpha)
    pd.Series(dist, name="diff_gross_mean").to_csv(OUT / f"label_placebo_k{k}.csv", index=False, float_format="%.6g")
bt = pd.concat({(k, s): pd.DataFrame(v, index=["mean", "lo", "hi"]).T * 1e4 for k, d in BOOT.items() for s, v in d.items()})
bt.index.names = ["k", "series", "block"]
print(f"block bootstrap, bps per day, two-sided {100 * (1 - CFG.alpha):.1f}% intervals")
display(bt.round(3))
display(pd.DataFrame(PLACEBO).T.round(5))
display(pd.DataFrame(G))

In [ ]:
summary = {"study": "P1-01", "synthetic": SYNTHETIC, "config": CFG.as_dict(), "years_data": YEARS_D,
           "window": [str(WIN[0].date()), str(WIN[-1].date())], "n_stocks": int(C.shape[1]),
           "terciles": TERC.to_dict(orient="index"), "table": {f"{k}|{t}": v for (k, t), v in TABLE.to_dict(orient="index").items()},
           "bootstrap_bps": {str(k): {s: {str(j): [1e4 * x for x in v] for j, v in d.items()} for s, d in dd.items()} for k, dd in BOOT.items()},
           "label_placebo": {str(k): v for k, v in PLACEBO.items()}, "gates": {str(k): v for k, v in G.items()},
           "versions": {"package": __version__, "alpaca": ALPACA_VERSION, "p1": p1.P1_VERSION}}
(OUT / "summary.json").write_text(json.dumps(summary, indent=2, default=float))
(OUT / "manifest.json").write_text(json.dumps({"files": sorted(p.name for p in OUT.iterdir()), "cache": str(CACHE)}, indent=2))
print(json.dumps({str(k): {x: g[x] for x in ("replication_verdict", "economic_verdict")} for k, g in G.items()}, indent=2))